In [ ]:
from chi import server, context, lease, network
import chi, os, time, datetime

context.version = "1.0"
context.choose_project()
context.choose_site(default="KVM@TACC")
username = os.getenv("USER")
print(username)

In [ ]:
l = lease.Lease(
    f"lease-ingest-{username}",
    duration=datetime.timedelta(hours=12)
)
l.add_flavor_reservation(
    id=chi.server.get_flavor_id("m1.xlarge"),
    amount=1
)
l.submit(idempotent=True)
l.show()

In [ ]:
import time

os_conn = chi.clients.connection()
cinder_client = chi.clients.cinder()

images_list = list(os_conn.image.images(name="CC-Ubuntu24.04"))
image_id = images_list[0].id
print("image_id:", image_id)

boot_vol = cinder_client.volumes.create(
    name=f"ingest-boot-{username}",
    size=150,
    imageRef=image_id,
)
print("volume id:", boot_vol.id)

while True:
    vol = cinder_client.volumes.get(boot_vol.id)
    print("status:", vol.status)
    if vol.status == "available":
        break
    time.sleep(10)

print("volume ready")

In [ ]:
bdm = [{
    "boot_index": 0,
    "uuid": boot_vol.id,
    "source_type": "volume",
    "destination_type": "volume",
    "delete_on_termination": True,
}]

srv = os_conn.compute.create_server(
    name=f"node-ingest-{username}",
    flavor_id=server.get_flavor_id(
        l.get_reserved_flavors()[0].name
    ),
    block_device_mapping_v2=bdm,
    networks=[{
        "uuid": os_conn.network.find_network("sharednet1").id
    }],
    key_name="id_rsa_chameleon",
)

print("waiting for ACTIVE...")
srv = os_conn.compute.wait_for_server(srv)
print("status:", srv.status)

In [ ]:
sharednet = os_conn.network.find_network("sharednet1")
port = next(
    p for p in os_conn.network.ports(device_id=srv.id)
    if p.network_id == sharednet.id
)
floating_net = os_conn.network.find_network("public")
fip = os_conn.network.create_ip(floating_network_id=floating_net.id)
os_conn.network.update_ip(fip, port_id=port.id)
print("floating IP:", fip.floating_ip_address)

security_groups = [
    {"name": "allow-ssh",  "port": 22,   "description": "SSH"},
    {"name": "allow-8888", "port": 8888, "description": "Jupyter"},
]

for sg in security_groups:
    secgroup = network.SecurityGroup({
        "name": sg["name"],
        "description": sg["description"],
    })
    secgroup.add_rule(direction="ingress", protocol="tcp", port=sg["port"])
    secgroup.submit(idempotent=True)
    os_conn.compute.add_security_group_to_server(srv, sg["name"])

print("security groups attached")

In [ ]:
s = server.get_server(f"node-ingest-{username}")
s.check_connectivity()

In [ ]:
s.execute("sudo mkdir -p /data/uhd-iqa /data/flickr-aes")
s.execute("sudo chown -R cc /data")
s.execute("pip install pandas numpy pyarrow tqdm pillow --break-system-packages")
s.execute("pip install git+https://github.com/openai/CLIP.git --break-system-packages")
s.execute("pip install torch torchvision --break-system-packages")
s.execute("pip install gdown --break-system-packages")
print("deps installed")

In [ ]:
s.execute("git clone https://github.com/binti-p/aesthetic-hub-data")
print("repo cloned")

In [ ]:
# this downloads flickr-aes from google drive and uhd-iqa from source
# flickr download includes a 6GB zip — takes a few minutes
result = s.execute("cd ~/aesthetic-hub-data && python3 pipelines/ingestion/download.py")
print(result)

In [ ]:
context.choose_project()
context.choose_site(default="CHI@TACC")

conn = chi.clients.connection()
project_id = conn.current_project_id
identity_ep = conn.session.get_endpoint(service_type="identity", interface="public")
url = f"{identity_ep}/v3/users/{conn.current_user_id}/credentials/OS-EC2"

resp = conn.session.post(url, json={"tenant_id": project_id})
resp.raise_for_status()
ec2 = resp.json()["credential"]

print("EC2 Access:", ec2["access"])
print("EC2 Secret:", ec2["secret"])

In [ ]:
access_key = ec2["access"]
secret_key  = ec2["secret"]

context.choose_site(default="KVM@TACC")

config = f"""[chi_tacc]
type = s3
provider = Ceph
access_key_id = {access_key}
secret_access_key = {secret_key}
endpoint = https://chi.tacc.chameleoncloud.org:7480
no_check_bucket = true
"""

s.execute("mkdir -p ~/.config/rclone")
s.execute(f"printf '{config}' > ~/.config/rclone/rclone.conf")
result = s.execute("rclone lsd chi_tacc:")
print(result)

In [ ]:
CONTAINER = "ObjStore_proj21"

s.execute(f"rclone copy /tmp/aesthetic-hub-output/global-uhd chi_tacc:{CONTAINER}/datasets/global-uhd --progress")
s.execute(f"rclone copy /tmp/aesthetic-hub-output/global-flickr chi_tacc:{CONTAINER}/datasets/global-flickr --progress")
s.execute(f"rclone copy /tmp/aesthetic-hub-output/personalized-flickr chi_tacc:{CONTAINER}/datasets/personalized-flickr --progress")
print("parquets uploaded")

In [ ]:
s.execute(f"rclone copy /data/uhd-iqa/uhd-iqa-metadata.csv chi_tacc:{CONTAINER}/raw-data/uhd-iqa/")
s.execute(f"rclone copy '/data/flickr-aes/FLICKR-AES_image_score.txt' chi_tacc:{CONTAINER}/raw-data/flickr-aes/")
s.execute(f"rclone copy '/data/flickr-aes/FLICKR-AES_image_labeled_by_each_worker.csv' chi_tacc:{CONTAINER}/raw-data/flickr-aes/")
print("metadata files uploaded")

In [ ]:
# uhd images (~6GB)
s.execute(f"rclone copy /data/uhd-iqa/training chi_tacc:{CONTAINER}/raw-data/uhd-iqa/images --progress")
s.execute(f"rclone copy /data/uhd-iqa/validation chi_tacc:{CONTAINER}/raw-data/uhd-iqa/images --progress")
s.execute(f"rclone copy /data/uhd-iqa/test chi_tacc:{CONTAINER}/raw-data/uhd-iqa/images --progress")

# flickr images (~6GB, takes a while)
s.execute(f"rclone copy /data/flickr-aes/40K chi_tacc:{CONTAINER}/raw-data/flickr-aes/images --progress")
print("images uploaded")

In [ ]:
result = s.execute("cd ~/aesthetic-hub-data && python3 pipelines/ingestion/write_metadata.py")
print(result)
result = s.execute(f"rclone copy /tmp/aesthetic-hub-metadata chi_tacc:{CONTAINER} --progress")
print(result)

In [ ]:
# the boot volume has delete_on_termination=True so it is deleted too
# data is safe in ObjStore_proj21 which is independent of the VM
context.choose_site(default="KVM@TACC")
s_teardown = server.get_server(f"node-ingest-{username}")
s_teardown.delete()
print("VM deleted")